# Elbow X-ray Abnormality Classification — MURA Dataset

## Task 0: Scope

- **Body part / anatomical region:** Elbow (musculoskeletal)
- **Imaging modality:** X-ray
- **Problem type:** Classification (binary — normal vs. abnormal)
- **Target class(es):** positive (abnormal) vs. negative (normal)
- **Dataset source:** MURA v1.1 (Stanford ML Group), elbow subset — https://stanfordmlgroup.github.io/competitions/mura/ — accessed:13/9/2026.
- **Real-world question:** Can an AI model flag likely abnormal elbow X-rays to help prioritize radiologist review?
- **Who would use it:** A radiologist.

## Task 0a: Dataset Sourcing

- Dataset: MURA v1.1, XR_ELBOW subset.
- License: Released by Stanford ML Group for non-commercial research/coursework use under their dataset agreement, accepted at registration.
- Free account required: Yes.
- Confirmed the elbow subset is large enough to split into train/validation/test without leakage.
- MURA v1.1, elbow subset, https://stanfordmlgroup.github.io/competitions/mura/, accessed on the 13/9/2026.

## A1. Problem Definition & Clinical Framing

**What this is:** A binary image classifier that takes a single elbow X-ray image and predicts whether it is "normal" (negative) or "abnormal" (positive) — abnormal meaning the radiologist who originally labeled the MURA study identified some abnormality (e.g. fracture, hardware, degenerative change) in the elbow.

**What each result means in plain language:**
- Negative / normal prediction: the model did not detect patterns consistent with the abnormal elbow X-rays it was trained on.
- Positive / abnormal prediction: the model detected patterns consistent with abnormal elbow X-rays. This does not identify what the abnormality is — just that something looked abnormal.


In [1]:
!pip install redivis

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import redivis

user = redivis.user("aimi")
dataset = user.dataset("mura_msk_xrays:cv1a:v1_0")
table = dataset.table("mura_v1_1:gcvw")

df = table.to_pandas_dataframe(max_results=10)
df.head()

  0%|          | 0/10 [00:00<?, ?it/s]

,file_id,file_name,size,added_at,md5_hash
0,gcvw-8sbb61dwy.gO5IC4xwWxAUUsDasTmUKg,valid/XR_SHOULDER/patient11735/study1_positive...,115318,2026-03-07 17:06:43.175000,ejnNN6MEckGAIZjC3zHl6w==
1,gcvw-8sbb61dwy.NH2qnAXQBhm55rM_goRkJg,valid/XR_SHOULDER/patient11738/study1_positive...,79333,2026-03-07 17:06:43.175000,XnHdh7XC3vvfw0sjDNR6Pw==
2,gcvw-8sbb61dwy.Kw2qYuOi4i4t3hzKFVGDbg,valid/XR_SHOULDER/patient11770/study1_negative...,56967,2026-03-07 17:06:43.175000,hGk5ZhYMC1+YqtZVwxJZGQ==
3,gcvw-8sbb61dwy.Ccz6BtJ7ZCW24r8dwPSsbQ,valid/XR_SHOULDER/patient11794/study1_negative...,43658,2026-03-07 17:06:43.175000,aJ1anfQwcg4QxhnkArXE9Q==
4,gcvw-8sbb61dwy.GPfYLrhLWxUKiCZkwQtkVQ,valid/XR_SHOULDER/patient11798/study1_negative...,54974,2026-03-07 17:06:43.175000,M+IKxg5R3BTb3ZErLmvIuw==


In [3]:
elbow_query = dataset.query("""
    SELECT * FROM `mura_v1_1:gcvw`
    WHERE file_name LIKE '%XR_ELBOW%'
""")

elbow_df = elbow_query.to_pandas_dataframe()
print(elbow_df.shape)
elbow_df.head()

  0%|          | 0/5396 [00:00<?, ?it/s]

(5396, 5)


,file_id,file_name,size,added_at,md5_hash
0,gcvw-8sbb61dwy.YVrReIak-sMswDgTaojjCg,valid/XR_ELBOW/patient11830/study1_positive/im...,102651,2026-03-07 17:06:56.962000,tGN/vdrZ820ns+cBGG5LMg==
1,gcvw-8sbb61dwy.Q8sDkX4hXTp5z3qfNu4ALg,valid/XR_ELBOW/patient11847/study1_negative/im...,147201,2026-03-07 17:06:56.962000,KUgeEHNur+ccyheBlg8eTg==
2,gcvw-8sbb61dwy.D0JucES4Q9opFOwUBpa0pA,valid/XR_ELBOW/patient11861/study1_negative/im...,45978,2026-03-07 17:06:56.962000,Rjr053ywMVO0zpsPm8rd8g==
3,gcvw-8sbb61dwy.biqQ_6YRP_BDyMP2wyAM3w,valid/XR_ELBOW/patient11880/study1_negative/im...,166543,2026-03-07 17:06:56.962000,CyBdcM+urVBdJ2lZYFD9Kg==
4,gcvw-8sbb61dwy.GwYFxsDFj-pncPjKw_pi9Q,valid/XR_ELBOW/patient11824/study2_negative/im...,167680,2026-03-07 17:06:56.962000,slq0SL4nnXEnnOic8Uslfw==


In [4]:
elbow_df['label'] = elbow_df['file_name'].apply(lambda x: 1 if 'positive' in x.lower() else 0)
print("Per-image counts:")
print(elbow_df['label'].value_counts())

Per-image counts:
label
0    3160
1    2236
Name: count, dtype: int64


In [5]:
import io
import sys
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from concurrent.futures import ThreadPoolExecutor

In [6]:
elbow_df['patient_id'] = elbow_df['file_name'].str.extract(r'(?P<patient_id>patient\d+)')
unique_patients = list(elbow_df['patient_id'].unique())

In [7]:
train_patients, temp_patients = train_test_split(unique_patients, test_size=0.30, random_state=42)
val_patients, test_patients = train_test_split(temp_patients, test_size=0.50, random_state=42)

In [8]:
train_df = elbow_df[elbow_df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df   = elbow_df[elbow_df['patient_id'].isin(val_patients)].reset_index(drop=True)
test_df  = elbow_df[elbow_df['patient_id'].isin(test_patients)].reset_index(drop=True)

print(f"Split Profiles -> Train images: {len(train_df)} | Val images: {len(val_df)} | Test images: {len(test_df)}")

Split Profiles -> Train images: 3801 | Val images: 805 | Test images: 790


In [9]:
IMG_SIZE = (224, 224)

def fetch_single_image(row):
    try:
        file_bytes = table.file(row['file_name']).read()
        img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
        img = img.resize(IMG_SIZE)
        return np.array(img) / 255.0
    except Exception as e:
        return None

def load_images(df, split_name):
    print(f"Fetching {split_name} data via parallel threads...")
    rows = [row for _, row in df.iterrows()]
    images = []
    
    with ThreadPoolExecutor(max_workers=16) as executor:
        for count, img_array in enumerate(executor.map(fetch_single_image, rows)):
            if img_array is not None:
                images.append(img_array)
            if (count + 1) % 50 == 0 or (count + 1) == len(df):
                sys.stdout.write(f"Progress: {count + 1}/{len(df)} fetched")
                sys.stdout.flush()
                
    print(f"Finished downloading {split_name} split.")
    return np.array(images), df['label'].values



In [2]:
X_train, y_train = load_images(train_df, "Train")
X_val, y_val = load_images(val_df, "Validation")
X_test, y_test = load_images(test_df, "Test")


NameError: name 'load_images' is not defined